# TP Jour 2 — Modèle d'Exécution Spark, Jointures Multi-Datasets & Couche Silver (Delta Lake)

### Objectifs de la séance
1. Observer la **Lazy Evaluation** et distinguer transformations **Narrow** vs **Wide (Shuffle)** sur **Spark Web UI** (`http://localhost:4040`).
2. Profiler la qualité de vos tables **Bronze** (issues d'un ou **plusieurs datasets Kaggle**), nettoyer les anomalies, typer fortement les colonnes et harmoniser les **clés de jointure inter-sources**.
3. Mettre en œuvre une **jointure distribuée multi-datasets maîtrisée** (`BroadcastHashJoin` via `F.broadcast` pour un référentiel/dataset compact, ou `SortMergeJoin` entre deux grandes sources) **en contrôlant la cardinalité (absence d'explosion N-à-M)**.
4. Sauvegarder votre table analytique enrichie dans **`/lakehouse/silver/`** et expérimenter les garanties **ACID de Delta Lake** (`_delta_log`, **Schema Enforcement**, **`MERGE INTO`**, **Time Travel**).

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.storagelevel import StorageLevel
from pyspark.errors import AnalysisException
from delta.tables import DeltaTable

spark = (
    SparkSession.builder
    .appName("J2_Spark_Delta_Silver")
    .getOrCreate()
)

spark.conf.set("spark.sql.shuffle.partitions", "50")
print("Session Spark active — Spark UI : http://localhost:4040")

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c268e4b2-7d14-4a30-aa74-4d20df5d7a78;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central


	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central


	found io.delta#delta-kernel-api;4.4.1 in central
	found org.roaringbitmap#RoaringBitmap;0.9.25 in central
	found com.fasterxml.jackson.core#jackson-databind;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-annotations;2.13.5 in central
	found com.fasterxml.jackson.core#jackson-core;2.13.5 in central
	found com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 in central
	found org.roaringbitmap#shims;0.9.25 in central


	found io.delta#delta-kernel-defaults;4.4.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.2 in central
	found org.apache.parquet#parquet-hadoop;1.16.0 in central
	found org.apache.parquet#parquet-column;1.16.0 in central
	found org.apache.parquet#parquet-common;1.16.0 in central
	found org.apache.parquet#parquet-format-structures;1.16.0 in central
	found javax.annotation#javax.annotation-api;1.3.2 in central
	found org.apache.parquet#parquet-encoding;1.16.0 in central
	found org.locationtech.jts#jts-core;1.20.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found io.airlift#aircompressor;2.0.2 in central
	found commons-pool#commons-pool;1.6 in central
	found com.github.luben#zstd-jni;1.5.7-3 in central


	found org.apache.hadoop#hadoop-client-api;3.4.2 in central
	found commons-logging#commons-logging;1.3.0 in central
	found org.apache.parquet#parquet-jackson;1.16.0 in central
	found io.delta#delta-kernel-unitycatalog;4.4.1 in central
:: resolution report :: resolve 877ms :: artifacts dl 22ms
	:: modules in use:
	com.fasterxml.jackson.core#jackson-annotations;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-core;2.13.5 from central in [default]
	com.fasterxml.jackson.core#jackson-databind;2.13.5 from central in [default]
	com.fasterxml.jackson.datatype#jackson-datatype-jdk8;2.13.5 from central in [default]
	com.github.luben#zstd-jni;1.5.7-3 from central in [default]
	com.google.code.findbugs#jsr305;3.0.2 from central in [default]
	commons-logging#commons-logging;1.3.0 from central in [default]
	commons-pool#commons-pool;1.6 from central in [default]
	io.airlift#aircompressor;2.0.2 from central in [default]
	io.delta#delta-kernel-api;4.4.1 from central in [default]
	

26/11/17 11:42:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/opt/venv/lib/python3.15/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/11/17 11:42:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Session Spark active — Spark UI : http://localhost:4040


## 1. Chargement de vos tables Bronze (Multi-Datasets / Multi-Tables) & Diagnostic Qualité

### Exercice 1.1 — Charger toutes vos tables Bronze et profiler valeurs manquantes / doublons
1. Chargez dans `bronze_dfs` votre table de faits principale ainsi que la (ou les) table(s) secondaire(s) à joindre depuis `/lakehouse/bronze/<nom_table>`.
2. Comptez le nombre de valeurs `NULL` sur vos colonnes clés (en particulier les **clés de jointure**) ainsi que les doublons éventuels.

In [2]:
BRONZE_TABLES: dict[str, str] = {
    name: f"/lakehouse/bronze/{name}"
    for name in ("flights", "airlines", "airports", "cancellation_codes")
}

bronze_dfs = {alias: spark.read.format("delta").load(path) for alias, path in BRONZE_TABLES.items()}
for alias, df in bronze_dfs.items():
    print(f"[{alias}] {BRONZE_TABLES[alias]} -> {df.count():,} lignes | colonnes : {df.columns[:8]}")

df_bronze = bronze_dfs["flights"]
quality_columns = ["year", "month", "day", "airline", "flight_number", "origin_airport", "destination_airport", "departure_delay", "arrival_delay", "cancellation_reason"]
null_report = df_bronze.select(*[F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in quality_columns])
print("Valeurs nulles dans les colonnes principales :")
null_report.show(truncate=False)

26/11/17 11:42:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[flights] /lakehouse/bronze/flights -> 5,819,079 lignes | colonnes : ['year', 'month', 'day', 'day_of_week', 'airline', 'flight_number', 'tail_number', 'origin_airport']


[airlines] /lakehouse/bronze/airlines -> 14 lignes | colonnes : ['iata_code', 'airline', '_ingested_at', '_source_dataset', '_source_file']


[airports] /lakehouse/bronze/airports -> 322 lignes | colonnes : ['iata_code', 'airport', 'city', 'state', 'country', 'latitude', 'longitude', '_ingested_at']


[cancellation_codes] /lakehouse/bronze/cancellation_codes -> 4 lignes | colonnes : ['cancellation_reason', 'cancellation_description', '_ingested_at', '_source_dataset', '_source_file']
Valeurs nulles dans les colonnes principales :


+----+-----+---+-------+-------------+--------------+-------------------+---------------+-------------+-------------------+
|year|month|day|airline|flight_number|origin_airport|destination_airport|departure_delay|arrival_delay|cancellation_reason|
+----+-----+---+-------+-------------+--------------+-------------------+---------------+-------------+-------------------+
|0   |0    |0  |0      |0            |0             |0                  |86153          |105071       |5729195            |
+----+-----+---+-------+-------------+--------------+-------------------+---------------+-------------+-------------------+



## 2. Harmonisation Inter-Datasets, Nettoyage & Jointure Distribuée (`BroadcastHashJoin` / `SortMergeJoin`)

### Bonnes pratiques de jointure entre plusieurs datasets hétérogènes :
1. **Harmonisation des clés de jointure (*Key Alignment*)** : deux datasets Kaggle d'auteurs différents n'ont presque jamais exactement le même format de clé :
   - *Axe temporel* : ramenez les horodatages à la même granularité et au même fuseau (`F.to_date(...)`, `F.date_trunc("hour", ...)` ou format `yyyy-MM`).
   - *Axe géographique / catégoriel* : normalisez la casse, les espaces et les codes (`F.upper(F.trim(col))`, code ISO pays/région, code postal ou grille `F.round(lat, 2)`).
2. **Prévention des explosions N-à-M** : avant un `LEFT JOIN` de la table principale (`N` lignes) vers un second dataset, assurez-vous que la clé de jointure est **unique côté table secondaire** via une pré-agrégation métier (`groupBy(join_keys).agg(...)`) ou `dropDuplicates(join_keys)` ! Vérifiez systématiquement que le nombre de lignes après `LEFT JOIN` n'explose pas.
3. **Choix de la stratégie physique Spark** :
   - **Table secondaire compacte (< quelques dizaines de Mo)** : forcez **`F.broadcast(df_sec)`** pour obtenir un `BroadcastHashJoin` sans *Shuffle* de la grande table.
   - **Deux grands datasets volumineux** : laissez Spark exécuter un `SortMergeJoin` distribué (optimisé dynamiquement par *Adaptive Query Execution*).

### Exercice 2.1 — Nettoyer, harmoniser vos clés, joindre vos sources et écrire `/lakehouse/silver/<table_silver>`
1. Nettoyez vos tables (`dropDuplicates`, filtrage d'anomalies, typage fort `F.to_timestamp` / `F.to_date` / numériques) et harmonisez la clé de jointure commune.
2. Créez au moins **2 nouvelles colonnes métier dérivées** (dont idéalement un indicateur combinant vos deux sources !) utiles pour les Jours 3 & 4.
3. Réalisez la jointure (avec `F.broadcast(...)` sur une table de dimension/dataset secondaire compact), contrôlez le **taux de rapprochement (*match rate*)** ainsi que l'absence de duplication N-M, et affichez `.explain("simple")`.
4. Écrivez la table enrichie au format **Delta** dans **`/lakehouse/silver/<votre_table>`**.

Pour les vols : une date invalide ou hors de 1900–2100, une clé ou un numéro de vol manquant, une distance négative ou un indicateur annulé/dérouté invalide entraîne le rejet de la ligne. Les vols annulés et les retards inconnus sont conservés. Les doublons sont les lignes identiques sur toutes les colonnes métier normalisées, hors métadonnées d'ingestion ; `flight_id` est le SHA-256 de ces colonnes. Les clés des référentiels doivent être uniques après normalisation.

Attention : chaque exécution réécrit `/lakehouse/silver/flights_enriched` et `/lakehouse/silver/j2_demo_delta`.

In [3]:
SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"

narrow = df_bronze.select("airline", "arrival_delay").filter(F.col("arrival_delay").isNotNull())
wide = narrow.groupBy("airline").agg(F.avg("arrival_delay").alias("mean_arrival_delay"))
print("Transformations créées sans action (lazy evaluation).")
print("Plan narrow :")
narrow.explain("simple")
print("Plan wide : l'Exchange correspond au shuffle de l'agrégation.")
wide.explain("simple")
wide.show(5, truncate=False)

def clean_key(column):
    return F.upper(F.trim(F.col(column)))

def reference_table(df, key, columns):
    ref = df.select(clean_key(key).alias(key), *[F.col(c) for c in columns]).filter(F.col(key).isNotNull() & (F.col(key) != "")).dropDuplicates()
    duplicates = ref.groupBy(key).count().filter(F.col("count") > 1).count()
    if duplicates:
        raise ValueError(f"Clés ambiguës dans le référentiel {key} : {duplicates}")
    return ref

airlines = reference_table(bronze_dfs["airlines"], "iata_code", ["airline"]).select(F.col("iata_code").alias("airline_key"), F.col("airline").alias("airline_name"))
airports = reference_table(bronze_dfs["airports"], "iata_code", ["airport", "city", "state", "country", "latitude", "longitude"])
cancellations = reference_table(bronze_dfs["cancellation_codes"], "cancellation_reason", ["cancellation_description"]).select(F.col("cancellation_reason").alias("cancel_key"), "cancellation_description")

df_clean = df_bronze
for column in ("year", "month", "day", "day_of_week", "flight_number", "departure_delay", "arrival_delay", "distance", "cancelled", "diverted"):
    df_clean = df_clean.withColumn(column, F.expr(f"try_cast(`{column}` as int)"))
for column in ("airline", "origin_airport", "destination_airport", "cancellation_reason"):
    df_clean = df_clean.withColumn(column, clean_key(column))
date_text = F.format_string("%04d-%02d-%02d", F.col("year"), F.col("month"), F.col("day"))
df_clean = df_clean.withColumn("flight_date", F.try_to_timestamp(date_text, F.lit("yyyy-MM-dd")).cast("date"))
valid = F.col("flight_date").isNotNull() & F.col("year").between(1900, 2100) & F.col("flight_number").isNotNull()
for column in ("airline", "origin_airport", "destination_airport"):
    valid = valid & F.col(column).isNotNull() & (F.col(column) != "")
valid = valid & (F.col("distance").isNull() | (F.col("distance") >= 0))
valid = valid & (F.col("cancelled").isNull() | F.col("cancelled").isin(0, 1)) & (F.col("diverted").isNull() | F.col("diverted").isin(0, 1))
bronze_count = df_bronze.count()
df_clean = df_clean.filter(valid)
valid_count = df_clean.count()
source_columns = [c for c in df_bronze.columns if not c.startswith("_")]
df_clean = df_clean.withColumn("flight_id", F.sha2(F.to_json(F.struct(*[F.col(c) for c in source_columns]), {"ignoreNullFields": "false"}), 256))
df_unique = df_clean.dropDuplicates(["flight_id"]).persist(StorageLevel.DISK_ONLY)

try:
    count_before_join = df_unique.count()
    print(f"Lignes Bronze : {bronze_count:,} | invalides : {bronze_count - valid_count:,} | doublons exacts : {valid_count - count_before_join:,}")
    airport_details = ("city", "state", "country", "latitude", "longitude")
    origin = airports.select(F.col("iata_code").alias("origin_key"), F.col("airport").alias("origin_airport_name"), *[F.col(c).alias(f"origin_{c}") for c in airport_details])
    destination = airports.select(F.col("iata_code").alias("destination_key"), F.col("airport").alias("destination_airport_name"), *[F.col(c).alias(f"destination_{c}") for c in airport_details])
    df_silver = (
        df_unique.join(F.broadcast(airlines), df_unique.airline == airlines.airline_key, "left").drop("airline_key")
        .join(F.broadcast(origin), F.col("origin_airport") == F.col("origin_key"), "left").drop("origin_key")
        .join(F.broadcast(destination), F.col("destination_airport") == F.col("destination_key"), "left").drop("destination_key")
        .join(F.broadcast(cancellations), F.col("cancellation_reason") == F.col("cancel_key"), "left").drop("cancel_key")
        .withColumn("is_arrival_delayed_15min", F.when(F.col("arrival_delay").isNotNull(), F.col("arrival_delay") >= 15))
        .withColumn("delay_per_100_miles", F.when(F.col("distance") > 0, F.col("arrival_delay") * 100.0 / F.col("distance")))
    )
    print("Plan de jointure :")
    df_silver.explain("simple")
    df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(SILVER_MAIN_PATH)
finally:
    df_unique.unpersist()

silver = spark.read.format("delta").load(SILVER_MAIN_PATH)
silver_count = silver.count()
assert silver_count == count_before_join, f"Cardinalité modifiée : {count_before_join:,} avant, {silver_count:,} après jointure."
print(f"Table Silver : {silver_count:,} lignes ; cardinalité préservée.")
silver.select(*[F.round(F.avg(F.col(c).isNotNull().cast("double")) * 100, 2).alias(c + "_match_pct") for c in ("airline_name", "origin_airport_name", "destination_airport_name")]).show(truncate=False)
silver.filter(F.col("cancelled") == 1).select(F.round(F.avg(F.col("cancellation_description").isNotNull().cast("double")) * 100, 2).alias("cancellation_match_pct")).show(truncate=False)

Transformations créées sans action (lazy evaluation).
Plan narrow :


== Physical Plan ==
*(1) Filter isnotnull(arrival_delay#22)
+- *(1) ColumnarToRow
   +- FileScan parquet [airline#4,arrival_delay#22] Batched: true, DataFilters: [isnotnull(arrival_delay#22)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[hdfs://namenode:9000/lakehouse/bronze/flights], PartitionFilters: [], PushedFilters: [IsNotNull(arrival_delay)], ReadSchema: struct<airline:string,arrival_delay:int>


Plan wide : l'Exchange correspond au shuffle de l'agrégation.


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[airline#4], functions=[avg(arrival_delay#22)])
   +- Exchange hashpartitioning(airline#4, 50), ENSURE_REQUIREMENTS, [plan_id=562]
      +- HashAggregate(keys=[airline#4], functions=[partial_avg(arrival_delay#22)])
         +- Filter isnotnull(arrival_delay#22)
            +- FileScan parquet [airline#4,arrival_delay#22] Batched: true, DataFilters: [isnotnull(arrival_delay#22)], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[hdfs://namenode:9000/lakehouse/bronze/flights], PartitionFilters: [], PushedFilters: [IsNotNull(arrival_delay)], ReadSchema: struct<airline:string,arrival_delay:int>




+-------+-------------------+
|airline|mean_arrival_delay |
+-------+-------------------+
|MQ     |6.457873460764516  |
|UA     |5.431593935741549  |
|NK     |14.471799501705833 |
|AA     |3.4513721447256764 |
|DL     |0.18675361236390797|
+-------+-------------------+
only showing top 5 rows


Lignes Bronze : 5,819,079 | invalides : 0 | doublons exacts : 0


Plan de jointure :


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [year#2418, month#2419, day#2420, day_of_week#2421, airline#2428, flight_number#2422, tail_number#6, origin_airport#2429, destination_airport#2430, scheduled_departure#9, departure_time#10, departure_delay#2423, taxi_out#12, wheels_off#13, scheduled_time#14, elapsed_time#15, air_time#16, distance#2425, wheels_on#18, taxi_in#19, scheduled_arrival#20, arrival_time#21, arrival_delay#2424, diverted#2427, cancelled#2426, ... 27 more fields]
   +- BroadcastHashJoin [cancellation_reason#2431], [cancel_key#2417], LeftOuter, BuildRight, false, false
      :- Project [year#2418, month#2419, day#2420, day_of_week#2421, airline#2428, flight_number#2422, tail_number#6, origin_airport#2429, destination_airport#2430, scheduled_departure#9, departure_time#10, departure_delay#2423, taxi_out#12, wheels_off#13, scheduled_time#14, elapsed_time#15, air_time#16, distance#2425, wheels_on#18, taxi_in#19, scheduled_arrival#20, arrival_time#21, 

Table Silver : 5,819,079 lignes ; cardinalité préservée.


+----------------------+-----------------------------+----------------------------------+
|airline_name_match_pct|origin_airport_name_match_pct|destination_airport_name_match_pct|
+----------------------+-----------------------------+----------------------------------+
|100.0                 |91.65                        |91.65                             |
+----------------------+-----------------------------+----------------------------------+



+----------------------+
|cancellation_match_pct|
+----------------------+
|100.0                 |
+----------------------+



## 3. Garanties Transactionnelles ACID de Delta Lake

### Exercice 3.1 — Explorer `_delta_log` et vérifier le *Schema Enforcement*
1. Lisez et affichez les premières lignes du fichier JSON de transaction `00000000000000000000.json` situé dans `{SILVER_MAIN_PATH}/_delta_log/`.
2. Sur une petite table de démonstration séparée, tentez d'insérer (`mode("append")`) un DataFrame au schéma incompatible et interceptez l'exception levée par Delta Lake, sans modifier la table analytique.

In [4]:
spark.read.text(f"{SILVER_MAIN_PATH}/_delta_log/00000000000000000000.json").show(8, truncate=110)

# La démonstration transactionnelle utilise une petite copie pour préserver la table analytique.
SILVER_DEMO_PATH = "/lakehouse/silver/j2_demo_delta"
sample_df = silver.select("flight_id", "flight_date", "airline", "arrival_delay")
sample_rows = sample_df.limit(3).collect()
assert len(sample_rows) == 3, "Au moins trois vols sont nécessaires pour la démonstration Delta."
demo_initial = spark.createDataFrame(sample_rows[:2], sample_df.schema).withColumn("demo_status", F.lit("initial"))
demo_initial.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(SILVER_DEMO_PATH)
bad_df = spark.createDataFrame([("test_1", "colonne_intruse")], ["fake_id", "intruder_col"])
try:
    bad_df.write.format("delta").mode("append").save(SILVER_DEMO_PATH)
except AnalysisException as error:
    if "schema" not in str(error).lower():
        raise
    print("Schema Enforcement validé : écriture incompatible refusée.")
else:
    raise AssertionError("L'écriture avec un schéma incompatible aurait dû échouer.")

+--------------------------------------------------------------------------------------------------------------+
|                                                                                                         value|
+--------------------------------------------------------------------------------------------------------------+
|{"commitInfo":{"timestamp":1794915962046,"operation":"WRITE","operationParameters":{"mode":"Overwrite","par...|
|{"metaData":{"id":"1895f4ad-75f0-4aa4-a5fd-c81a1272a67e","format":{"provider":"parquet","options":{}},"sche...|
|                                                      {"protocol":{"minReaderVersion":1,"minWriterVersion":2}}|
|{"add":{"path":"part-00000-616ffb23-18f9-43c9-8c3d-564795665804-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00001-6cb36dcd-c312-41a5-94e2-339b432bdb52-c000.snappy.parquet","partitionValues":{},"...|
|{"add":{"path":"part-00002-f9618803-128a-4da9-9459-8a4cd10ce3ea-c000.snappy.parquet","partition

Schema Enforcement validé : écriture incompatible refusée.


### Exercice 3.2 — Upsert (`MERGE INTO`) et Voyage dans le Temps (`VERSION AS OF`)
1. Prélevez trois vols de la table Silver et créez une petite table Delta de démonstration avec les deux premiers. Appliquez un **`MERGE INTO`** : mise à jour du deuxième et insertion du troisième.
2. Affichez l'historique et comparez la version avant le MERGE à la version courante. La table analytique principale reste inchangée.

In [5]:
target_delta = DeltaTable.forPath(spark, SILVER_DEMO_PATH)
version_before = target_delta.history(1).select("version").first()[0]
updates = spark.createDataFrame(sample_rows[1:], sample_df.schema).withColumn("demo_status", F.lit("merge"))
(target_delta.alias("target").merge(updates.alias("source"), "target.flight_id = source.flight_id")
    .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

history_df = target_delta.history()
history_df.select("version", "timestamp", "operation", "operationMetrics").show(truncate=80)
before = spark.read.format("delta").option("versionAsOf", version_before).load(SILVER_DEMO_PATH)
after = spark.read.format("delta").load(SILVER_DEMO_PATH)
before_status = {row.flight_id: row.demo_status for row in before.select("flight_id", "demo_status").collect()}
after_status = {row.flight_id: row.demo_status for row in after.select("flight_id", "demo_status").collect()}
assert len(before_status) == 2 and len(after_status) == 3
assert before_status[sample_rows[1].flight_id] == "initial" and after_status[sample_rows[1].flight_id] == "merge"
assert sample_rows[2].flight_id not in before_status and after_status[sample_rows[2].flight_id] == "merge"
print(f"Time Travel : version {version_before} = 2 lignes ; version courante = 3 lignes (1 mise à jour, 1 insertion).")

26/11/17 11:46:26 WARN MapPartitionsRDD: RDD 382 was locally checkpointed, its lineage has been truncated and cannot be recomputed after unpersisting


+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|version|              timestamp|operation|                                                                operationMetrics|
+-------+-----------------------+---------+--------------------------------------------------------------------------------+
|      1|2026-11-17 11:46:24.506|    MERGE|{numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetBytesRemoved -...|
|      0|2026-11-17 11:46:16.836|    WRITE|{numFiles -> 3, numRemovedFiles -> 0, numRemovedBytes -> 0, numOutputRows -> ...|
+-------+-----------------------+---------+--------------------------------------------------------------------------------+



Time Travel : version 0 = 2 lignes ; version courante = 3 lignes (1 mise à jour, 1 insertion).
